# Step 4 & 5: SQL Analytics - Joins, Aggregations, Window Functions, CTEs

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/ecommerce.db')


## Total revenue per customer

In [2]:
pd.read_sql("""
SELECT c.customer_id, c.name, ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY c.customer_id, c.name
ORDER BY revenue DESC
LIMIT 10
""", conn)


,customer_id,name,revenue
0,174,Sandra Davis,22741.62
1,89,Mary Peck,21058.03
2,80,Pamela Lopez,18470.00
3,200,Patricia Morrow,17466.51
4,120,Michael Craig,17350.17
5,71,Jeffrey Wood,17164.77
6,90,Charles Brown,16929.29
7,70,Lori Garcia,16881.99
8,161,Robert Monroe,16481.27
9,20,Joseph Martinez,16321.63


## Total revenue per category

In [3]:
pd.read_sql("""
SELECT p.category, ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id
GROUP BY p.category
ORDER BY revenue DESC
""", conn)


,category,revenue
0,Clothing,320860.49
1,Electronics,298486.70
2,Sports,231680.91
3,Toys,209960.62
4,Home,203714.68
5,Books,104981.81
6,Unknown,80320.15


## Total revenue per month

In [4]:
pd.read_sql("""
SELECT strftime('%Y-%m', o.order_date) AS month, ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY month
ORDER BY month
""", conn)


,month,revenue
0,2024-07,36050.40
1,2024-08,49655.59
2,2024-09,55827.24
3,2024-10,57706.62
4,2024-11,49379.79
5,2024-12,56602.13
6,2025-01,57879.39
7,2025-02,46339.12
8,2025-03,73189.32
9,2025-04,62680.11


## Top products by quantity sold and revenue

In [5]:
pd.read_sql("""
SELECT p.product_name, SUM(oi.quantity) AS units_sold,
       ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id
GROUP BY p.product_name
ORDER BY revenue DESC
LIMIT 10
""", conn)


,product_name,units_sold,revenue
0,Ok Lite,113,37706.29
1,Morning Lite,127,34721.27
2,Put,136,34400.27
3,Floor Pro,128,33742.12
4,Car Max,122,33686.61
5,Push Pro,107,33413.04
6,Them,131,33102.62
7,Yeah Pro,115,32311.36
8,Prove,117,30691.90
9,None Max,122,30576.98


## Average order value by customer segment

In [6]:
pd.read_sql("""
SELECT c.segment, ROUND(AVG(order_total), 2) AS avg_order_value
FROM customers c
JOIN (
    SELECT o.order_id, o.customer_id, SUM(oi.quantity * oi.unit_price) AS order_total
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.order_id
) t ON c.customer_id = t.customer_id
GROUP BY c.segment
""", conn)


,segment,avg_order_value
0,New,2073.22
1,Premium,1945.29
2,Regular,1965.36


## Rank customers by lifetime value (RANK / DENSE_RANK)

In [7]:
pd.read_sql("""
WITH customer_revenue AS (
    SELECT c.customer_id, c.name, SUM(oi.quantity * oi.unit_price) AS ltv
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY c.customer_id, c.name
)
SELECT customer_id, name, ROUND(ltv, 2) AS ltv,
       RANK() OVER (ORDER BY ltv DESC) AS ltv_rank,
       DENSE_RANK() OVER (ORDER BY ltv DESC) AS ltv_dense_rank
FROM customer_revenue
ORDER BY ltv_rank
LIMIT 10
""", conn)


,customer_id,name,ltv,ltv_rank,ltv_dense_rank
0,174,Sandra Davis,22741.62,1,1
1,89,Mary Peck,21058.03,2,2
2,80,Pamela Lopez,18470.00,3,3
3,200,Patricia Morrow,17466.51,4,4
4,120,Michael Craig,17350.17,5,5
5,71,Jeffrey Wood,17164.77,6,6
6,90,Charles Brown,16929.29,7,7
7,70,Lori Garcia,16881.99,8,8
8,161,Robert Monroe,16481.27,9,9
9,20,Joseph Martinez,16321.63,10,10


## Running total and 3-month moving average of revenue

In [8]:
pd.read_sql("""
WITH monthly_revenue AS (
    SELECT strftime('%Y-%m', o.order_date) AS month,
           SUM(oi.quantity * oi.unit_price) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY month
)
SELECT month, ROUND(revenue, 2) AS revenue,
       ROUND(SUM(revenue) OVER (ORDER BY month), 2) AS running_total,
       ROUND(AVG(revenue) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS moving_avg_3mo
FROM monthly_revenue
ORDER BY month
""", conn)


,month,revenue,running_total,moving_avg_3mo
0,2024-07,36050.40,36050.40,36050.40
1,2024-08,49655.59,85705.99,42852.99
2,2024-09,55827.24,141533.23,47177.74
3,2024-10,57706.62,199239.85,54396.48
4,2024-11,49379.79,248619.64,54304.55
5,2024-12,56602.13,305221.77,54562.85
6,2025-01,57879.39,363101.16,54620.44
7,2025-02,46339.12,409440.28,53606.88
8,2025-03,73189.32,482629.60,59135.94
9,2025-04,62680.11,545309.71,60736.18


## Monthly revenue growth rate (CTE with LAG)

In [9]:
pd.read_sql("""
WITH monthly_revenue AS (
    SELECT strftime('%Y-%m', o.order_date) AS month,
           SUM(oi.quantity * oi.unit_price) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY month
),
with_prev AS (
    SELECT month, revenue,
           LAG(revenue) OVER (ORDER BY month) AS prev_revenue
    FROM monthly_revenue
)
SELECT month, ROUND(revenue, 2) AS revenue,
       ROUND(prev_revenue, 2) AS prev_revenue,
       ROUND((revenue - prev_revenue) * 100.0 / prev_revenue, 2) AS growth_rate_pct
FROM with_prev
ORDER BY month
""", conn)


,month,revenue,prev_revenue,growth_rate_pct
0,2024-07,36050.40,NaN,NaN
1,2024-08,49655.59,36050.40,37.74
2,2024-09,55827.24,49655.59,12.43
3,2024-10,57706.62,55827.24,3.37
4,2024-11,49379.79,57706.62,-14.43
5,2024-12,56602.13,49379.79,14.63
6,2025-01,57879.39,56602.13,2.26
7,2025-02,46339.12,57879.39,-19.94
8,2025-03,73189.32,46339.12,57.94
9,2025-04,62680.11,73189.32,-14.36


In [10]:
conn.close()
